# Joke Chatbot

A command-line chatbot built in Python for the *Utveckling med Python, grund* course final project.

The chatbot can:
- Greet the user and recognize when someone introduces themselves (e.g. "my name is Fatima" or "I'm Fatima")
- Tell a random joke fetched live from a public joke API
- Save every conversation to a JSON file and reload past conversations the next time it runs
- Keep count of how many jokes it has told during a session, via a specialized subclass (`SuperChatBot`)

**GitHub repository:** https://github.com/TehFat/joke-chatbot-python


## Imports

This project uses three libraries:
- `requests` — an external library used to call the joke API over the internet
- `json` — a standard library module used to read and write conversation history as JSON
- `datetime` — a standard library module used to timestamp each message


In [ ]:
import requests
import json
from datetime import datetime


## The `ChatBot` class

This class is the blueprint for the chatbot. It stores the bot's name and conversation history as attributes, and defines methods for:
- greeting the user
- loading and saving conversation history to a JSON file (with `try`/`except` error handling)
- recording each exchange in the history
- fetching a random joke from an external API (with specific `try`/`except` error handling for connection errors, timeouts, and other request errors)
- deciding how to respond to a message, using `if`/`elif`/`else`

**How `respond()` decides what to say:** it checks the message against several conditions in order, and uses the *first* one that matches — for example, `"my name is"` is checked before the more general `"joke"` check, so a message like `"my name is Joker"` is treated as an introduction, not a joke request. Once it recognizes an introduction (`"my name is"` or `"i'm"`/`"i am"`), it also invites the user to ask for a joke, the same way the greeting branch does.

**Why `get_joke()`'s error handling is split into three cases:** a `ConnectionError` (no internet / can't reach the API), a `Timeout` (the API took too long to respond), and a general `RequestException` (anything else that can go wrong with the request) are handled separately, each with its own message, instead of one generic `except Exception`. The more specific exceptions have to be listed *before* the general one, since Python checks each `except` in order and stops at the first match.


In [ ]:
class ChatBot:
    def __init__(self, name):
        self.name = name
        self.history = []

    def greet(self):
        print(f"Hello! My name is {self.name}. How can I assist you today?")

    # Function to load chat history from a JSON file
    def load_history(self):
        try:
            with open("chat_history.json", "r") as file:
                self.history = json.load(file)
        except FileNotFoundError:
            self.history = []

    def save_history(self):
        with open("chat_history.json", "w") as file:
            json.dump(self.history, file, indent=4)

    def add_to_history(self, user_message, bot_reply):
        entry = {
            "time": str(datetime.now()),
            "you": user_message,
            "bot": bot_reply
        }
        self.history.append(entry)

    # Function to fetch a random joke from an API
    def get_joke(self):
        try:
            response = requests.get("https://official-joke-api.appspot.com/random_joke")
            if response.status_code == 200:
                joke_data = response.json()
                return f"{joke_data['setup']} ... {joke_data['punchline']}"
            else:
                return "Sorry, I couldn't fetch a joke at the moment."
        except requests.exceptions.ConnectionError:
            return "Sorry, I couldn't connect to the joke service. Please check your internet connection."
        except requests.exceptions.Timeout:
            return "Sorry, the request timed out. Please try again later."
        except requests.exceptions.RequestException as e:
            return f"An error occurred: {e}"

    def respond(self, message):
        message = message.lower()
        if "my name is" in message:
            name = message.split("my name is")[-1].strip()
            return f"Nice to meet you, {name.title()}! Want to hear a joke?"
        elif "i'm" in message or "i am" in message:
            name = message.split("i am")[-1].strip() if "i am" in message else message.split("i'm")[-1].strip()
            return f"Nice to meet you, {name.title()}! Want to hear a joke?"
        elif "joke" in message:
            return self.get_joke()
        elif "hi" in message or "hello" in message:
            return f"Hey there! I'm {self.name}. Want to hear a joke?"
        else:
            return f"You said: '{message}'. I'm here to help!"


## The `SuperChatBot` subclass

This class **inherits** from `ChatBot`, meaning it automatically gets everything `ChatBot` already does. It reuses the parent class's logic via `super()`, and adds one new feature on top: counting how many jokes it has told during the session.

- `super().__init__(name)` reuses the parent's constructor instead of repeating `self.name = name` and `self.history = []`
- `get_joke` is **overridden**: it calls `super().get_joke()` to reuse the original API-calling logic, then adds one line to increase the joke counter
- `respond` is also overridden: it checks for `"how many jokes"` first, then falls back to `super().respond(message)` for everything else


In [ ]:
# Subclass that extends ChatBot to keep track of the number of jokes told
class SuperChatBot(ChatBot):
    def __init__(self, name):
        super().__init__(name)
        self.joke_told = 0

    def get_joke(self):
        joke = super().get_joke()
        self.joke_told += 1
        return joke

    def joke_count(self):
        return f"I have told you {self.joke_told} joke(s) so far!"

    def respond(self, message):
        message = message.lower()
        if "how many jokes" in message:
            return self.joke_count()
        else:
            return super().respond(message)


## Running the chatbot

This creates a `SuperChatBot`, loads any previously saved conversation history, greets the user, and then starts a loop that keeps chatting until the user types `exit` or `quit` — at which point the conversation is saved to `chat_history.json`.

Run this cell to start chatting. Type your messages in the input box that appears, and type `exit` when you're done.


In [ ]:
bot = SuperChatBot("Jokey")
bot.load_history()
bot.greet()

while True:
    message = input("You: ")
    if message.lower() in ["exit", "quit"]:
        bot.save_history()
        print("Goodbye! Have a great day!")
        break
    else:
        reply = bot.respond(message)
        print(f"{bot.name}: {reply}")
        bot.add_to_history(message, reply)


## Reflection

Working on this project helped me understand object-oriented programming in a much more concrete way than just reading about it — actually building a class, then extending it with inheritance and seeing `super()` reuse the parent's code, made the concept click in a way theory alone hadn't. The most challenging part was getting comfortable with Python's indentation rules; several bugs during development came from a method being indented at the wrong level (for example, ending up nested inside `__init__` instead of being a proper method of the class), and learning to read those errors carefully taught me to pay much closer attention to code structure. Another good lesson was realizing that my early tests often only checked the "success" path of an `if/else` and never the other branch, which meant bugs could easily hide unnoticed — I now try to deliberately test both outcomes. If I continued developing this project, I would like to make the keyword-matching smarter and let the chatbot remember the user's name across sessions, not just within one conversation.

This also reflects a broader trend in the AI industry right now: chatbots are moving away from rule-based keyword matching toward language models that understand meaning rather than exact words. A natural next step for this project would be replacing the keyword-matching in `respond()` with a call to a language model API instead — the same underlying architecture (classes, error handling, saving state), just with a smarter decision-maker at the center.

For the industry analysis, certificate discussion, and full reflection, see this project's `README.md`.
